In [ ]:
!pip install datasets pandas torch "transformers[torch]" python-dotenv peft "torchao>=0.16.0"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "ஒரு நாள்"
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(inputs.input_ids, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


ஒரு நாள்்்்்்்்்்்்்்்்்்்்்்்்்்்்்்்்்்�


In [ ]:
from datasets import load_dataset
raw_data = load_dataset("tniranjan/aitamilnadu_tamil_stories_no_instruct",split="train[:1000]")
data = raw_data.train_test_split(train_size=0.95) # use 95% of data for training remaining 5% for testing
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 950
    })
    test: Dataset({
        features: ['text'],
        num_rows: 50
    })
})

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
def preprocess_batch(batch):
  return tokenizer(batch["text"], truncation=True, padding=True, max_length=200)

tokenized_dataset = data.map(
preprocess_batch,
batched=True,
batch_size =4,
remove_columns=data["train"].column_names,)
# Print dataset details
print(tokenized_dataset)

# tokenizes 4 rows at a time

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 950
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 50
    })
})


In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)
data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensors='pt', seed=None)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
model.train()

from torch.optim import AdamW
from transformers import TrainingArguments, Trainer

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-5)

# Training arguments
training_args = TrainingArguments(
    output_dir="./output",
    eval_strategy="epoch",
    save_steps=500,
    learning_rate=1e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=50,
    logging_dir="./logs",
    resume_from_checkpoint=True
)

# Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    args=training_args,
    optimizers=(optimizer, None),
    data_collator=data_collator
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,1.197056,1.106522
2,1.183420,1.087396
3,1.177887,1.082945


TrainOutput(global_step=1425, training_loss=1.1943231482254832, metrics={'train_runtime': 92.2053, 'train_samples_per_second': 30.909, 'train_steps_per_second': 15.455, 'total_flos': 145952686080000.0, 'train_loss': 1.1943231482254832, 'epoch': 3.0})

In [28]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# Save only the PEFT adapters and their configuration
model.save_pretrained("/content/drive/My Drive/fine_tuned_distilgpt2")
tokenizer.save_pretrained("/content/drive/My Drive/fine_tuned_distilgpt2")

('/content/drive/My Drive/fine_tuned_distilgpt2/tokenizer_config.json',
 '/content/drive/My Drive/fine_tuned_distilgpt2/tokenizer.json')

In [31]:
model = AutoModelForCausalLM.from_pretrained("/content/drive/My Drive/fine_tuned_distilgpt2")
model

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /content/drive/My Drive/fine_tuned_distilgpt2
Key                                                                                 | Status     | 
------------------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): lora.Linear(
            (base_layer): Conv1D(nf=2304, nx=768)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=768, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2304, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=Fals

In [32]:
text = "ஒரு நாள"
inputs = tokenizer(text, return_tensors="pt")
# Generate story
output = model.generate(inputs.input_ids, max_new_tokens=100)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


ஒரு நாளு நாளு நாளு நாளு நாளு நாளு நாளு நாளு ந�
